# Description of task 
### I observed that a large percentage of the CPDB is composed of ConA homologs.
### To identify all pdbs in the database with similarity to ConA,
### I'll TM-align each one to either pro-ConA (2FMD) or ConA (2CNA)

In [1]:
import pandas as pd
fp = "/home/ubuntu/CIRPIN/cpdb_comparison/foldseek_CPDB_comparison/cpdb_pdbs_2/sequence_alignment/CPDB_with_uniprotID.csv"
df = pd.read_csv(fp)

In [9]:
df[df['Protein1'] == '2cnaA']

,Protein1,Protein2,tmscore_max,tmscore_cp_max,tmscore_min,tmscore_cp_min,progres_score,cirpin_score,Protein1_sccs,Protein2_sccs,tm_score_min_difference,Protein1_uniprot,Protein2_uniprot
1602,2cnaA,2fmdA,0.48268,0.94310,0.48268,0.94310,0.991433,0.997148,b.29.1.1,b.29.1.0,0.46042,P02866,P42088
1603,2cnaA,1q8oA,0.47097,0.91826,0.46719,0.91088,0.993226,0.996786,b.29.1.1,b.29.1.1,0.44369,P02866,Q8GSD2
1604,2cnaA,1ciwA,0.46969,0.88655,0.46969,0.88655,0.992326,0.996076,b.29.1.1,b.29.1.1,0.41686,P02866,P02872
1605,2cnaA,1bzwA,0.47005,0.88745,0.47005,0.88745,0.992512,0.996899,b.29.1.1,b.29.1.1,0.41740,P02866,P02872
1606,2cnaA,1n47A,0.46058,0.90338,0.46058,0.90338,0.991040,0.994550,b.29.1.1,b.29.1.1,0.44280,P02866,P56625
1607,2cnaA,1fnyA,0.47222,0.90533,0.47222,0.90533,0.992276,0.995318,b.29.1.1,b.29.1.1,0.43311,P02866,Q41159
1608,2cnaA,1g7yA,0.49071,0.90678,0.46463,0.85809,0.990931,0.994389,b.29.1.1,b.29.1.1,0.39346,P02866,P19588
1609,2cnaA,1g8wA,0.46406,0.89485,0.46406,0.89485,0.989858,0.995986,b.29.1.1,b.29.1.1,0.43079,P02866,P05087
1610,2cnaA,1bjqA,0.49483,0.90347,0.46861,0.85493,0.987151,0.994847,b.29.1.1,b.29.1.1,0.38632,P02866,P05045
1611,2cnaA,1uzyA,0.46736,0.90309,0.46547,0.89948,0.990271,0.995087,b.29.1.1,b.29.1.1,0.43401,P02866,Q6YD91


In [2]:
pdbs = list(pd.concat([df['Protein1'], df['Protein2']]).unique())


In [5]:
def convert_name(name):
    '''Convert name to filename used:
    ex. 1fp3A --> 1FP3_A
    '''
    if pd.isna(name) or len(name) < 5:
        return name
    # Split at position 4 (before chain identifier)
    pdb_code = name[:4].upper()
    chain = name[4:]
    fixed_name = f"{pdb_code}_{chain}.pdb"
    return fixed_name

pdbs_converted = [convert_name(p) for p in pdbs]


In [6]:
import os 
def tmscore(q, t, cp=False):
    '''Run TM-align and get back TM-align score'''
    if cp:
        output = os.popen(f'/home/ubuntu/TM_tools/TMalign {q} {t} -cp')
    else:
        output = os.popen(f'/home/ubuntu/TM_tools/TMalign {q} {t}')
    
    tms = {"tms": []}
    parse_float = lambda x: float(x.split("=")[1].split()[0])
    
    for line in output:
        line = line.rstrip()
        if line.startswith("TM-score"): 
            tms["tms"].append(parse_float(line))
    
    if tms['tms']:
        max_tms = max(tms['tms'])
    else:
        print(f"Warning: tms['tms'] is empty, setting min_tms to 0 for {q}, {t}")
        max_tms = 0
    
    return max_tms

In [23]:
df.head()

,Protein1,Protein2,tmscore_max,tmscore_cp_max,tmscore_min,tmscore_cp_min,progres_score,cirpin_score,Protein1_sccs,Protein2_sccs,tm_score_min_difference,Protein1_uniprot,Protein2_uniprot
0,1fp3A,1vd5A,0.67760,0.67760,0.67760,0.67760,0.939401,0.983705,a.102.1.3,a.102.1.7,0.00000,P17560,Q9RC92
1,1fp3A,1wu4A,0.67768,0.67887,0.67768,0.67887,0.971649,0.990784,a.102.1.3,a.102.1.2,0.00119,P17560,Q9KB30
2,1fp3A,2drqA,0.67558,0.67691,0.67558,0.67691,0.976586,0.990878,a.102.1.3,a.102.1.2,0.00133,P17560,Q9KB30
3,1fp3A,1wu6A,0.67434,0.67558,0.67434,0.67558,0.974675,0.990933,a.102.1.3,a.102.1.2,0.00124,P17560,Q9KB30
4,1fp3A,1hzfA,0.54695,0.55763,0.54695,0.55763,0.911674,0.948243,a.102.1.3,a.102.4.4,0.01068,P17560,P0C0L4


In [28]:

## Calculate TM score between every CPDB pdb and Pro-conA, ConA

import pandas as pd

proconA = '/home/ubuntu/CIRPIN/cpdb_comparison/foldseek_CPDB_comparison/cpdb_pdbs_2/2FMD_A.pdb'
conA = '/home/ubuntu/CIRPIN/cpdb_comparison/foldseek_CPDB_comparison/cpdb_pdbs_2/2CNA_A.pdb'
pdb_dir = '/home/ubuntu/CIRPIN/cpdb_comparison/foldseek_CPDB_comparison/cpdb_pdbs_2/'

# Initialize DataFrame with columns
conA_df = pd.DataFrame(columns=['pdb_id', 'conA_score', 'proconA_score'])

for pdb in pdbs_converted:
    pdb_path = pdb_dir + pdb
    conA_score = tmscore(conA, pdb_path)
    proconA_score = tmscore(proconA, pdb_path)
    # Add row to dataframe
    new_row = pd.DataFrame({
        'pdb_id': [pdb[:-4]], 
        'conA_score': [conA_score], 
        'proconA_score': [proconA_score]
    })
    conA_df = pd.concat([conA_df, new_row], ignore_index=True)
    if len(conA_df) % 100 == 0:
        print(f'Processed {len(conA_df)}')


/tmp/ipykernel_3953487/3958607151.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  conA_df = pd.concat([conA_df, new_row], ignore_index=True)


Processed 100
Processed 200
Processed 300
Processed 400


Sequence is too short <3!: /home/ubuntu/CIRPIN/cpdb_comparison/foldseek_CPDB_comparison/cpdb_pdbs_2/1TUC_A.pdb
Sequence is too short <3!: /home/ubuntu/CIRPIN/cpdb_comparison/foldseek_CPDB_comparison/cpdb_pdbs_2/1TUC_A.pdb


Processed 500
Processed 600
Processed 700
Processed 800
Processed 900


Sequence is too short <3!: /home/ubuntu/CIRPIN/cpdb_comparison/foldseek_CPDB_comparison/cpdb_pdbs_2/1MH1_A.pdb
Sequence is too short <3!: /home/ubuntu/CIRPIN/cpdb_comparison/foldseek_CPDB_comparison/cpdb_pdbs_2/1MH1_A.pdb


Sequence is too short <3!: /home/ubuntu/CIRPIN/cpdb_comparison/foldseek_CPDB_comparison/cpdb_pdbs_2/1RYF_A.pdb
Sequence is too short <3!: /home/ubuntu/CIRPIN/cpdb_comparison/foldseek_CPDB_comparison/cpdb_pdbs_2/1RYF_A.pdb


Processed 1000
Processed 1100


Sequence is too short <3!: /home/ubuntu/CIRPIN/cpdb_comparison/foldseek_CPDB_comparison/cpdb_pdbs_2/2H7V_A.pdb
Sequence is too short <3!: /home/ubuntu/CIRPIN/cpdb_comparison/foldseek_CPDB_comparison/cpdb_pdbs_2/2H7V_A.pdb


Processed 1200
Processed 1300
Processed 1400
Processed 1500
Processed 1600
Processed 1700
Processed 1800
Processed 1900
Processed 2000
Processed 2100
Processed 2200


In [29]:

### Get list of proconA/conA homologs by filtering for scores > 0.5 using max TM score
conA_homologs = conA_df[(conA_df['conA_score'] > 0.5) | (conA_df['proconA_score'] > 0.5)]['pdb_id'].to_list()
conA_homologs_lower = [name[:4].replace('_', '').lower() + name[-1] for name in conA_homologs]


In [44]:
conA_rows = df[df['Protein1'].isin(conA_homologs_lower) | df['Protein2'].isin(conA_homologs_lower)].index
df_no_conA = df.drop(conA_rows)

In [51]:
len(conA_rows)

876

In [46]:
df_no_conA.head()

,Protein1,Protein2,tmscore_max,tmscore_cp_max,tmscore_min,tmscore_cp_min,progres_score,cirpin_score,Protein1_sccs,Protein2_sccs,tm_score_min_difference,Protein1_uniprot,Protein2_uniprot
0,1fp3A,1vd5A,0.67760,0.67760,0.67760,0.67760,0.939401,0.983705,a.102.1.3,a.102.1.7,0.00000,P17560,Q9RC92
1,1fp3A,1wu4A,0.67768,0.67887,0.67768,0.67887,0.971649,0.990784,a.102.1.3,a.102.1.2,0.00119,P17560,Q9KB30
2,1fp3A,2drqA,0.67558,0.67691,0.67558,0.67691,0.976586,0.990878,a.102.1.3,a.102.1.2,0.00133,P17560,Q9KB30
3,1fp3A,1wu6A,0.67434,0.67558,0.67434,0.67558,0.974675,0.990933,a.102.1.3,a.102.1.2,0.00124,P17560,Q9KB30
4,1fp3A,1hzfA,0.54695,0.55763,0.54695,0.55763,0.911674,0.948243,a.102.1.3,a.102.4.4,0.01068,P17560,P0C0L4


In [49]:
df[(df['tmscore_cp_min'] < 0.5) & (df['tmscore_cp_max'] > 0.5)].head(30)

,Protein1,Protein2,tmscore_max,tmscore_cp_max,tmscore_min,tmscore_cp_min,progres_score,cirpin_score,Protein1_sccs,Protein2_sccs,tm_score_min_difference,Protein1_uniprot,Protein2_uniprot
15,1wzzA,1kktA,0.69636,0.69412,0.48420,0.48250,0.923056,0.971600,a.102.1.2,a.102.2.1,-0.00170,P37696,P31723
16,1wzzA,2h6gB,0.62184,0.56706,0.49232,0.45035,0.906623,0.971099,a.102.1.2,NaN,-0.04197,P37696,P49356
23,2gh4A,1gszA,0.69203,0.69203,0.42755,0.42755,0.847268,0.946206,a.102.1.6,a.102.4.2,0.00000,O34559,P33247
44,1a22A,1yuzA,0.43470,0.52277,0.39885,0.48143,0.728325,0.877635,a.26.1.1,g.41.5.1,0.08258,P01241,P30820
46,1a22A,1yuxA,0.43475,0.52275,0.39886,0.48144,0.719050,0.888328,a.26.1.1,g.41.5.1,0.08258,P01241,P30820
58,1a7dA,2bkcA,0.48649,0.56661,0.40322,0.46935,0.448996,0.773491,a.24.4.1,a.25.1.1,0.06613,P02247,P80725
98,1aluA,1z6oM,0.48509,0.57232,0.41122,0.48968,0.649516,0.944920,a.26.1.1,a.25.1.1,0.07846,P05231,A0A7E5WTY7
113,1ax8A,2g2dA,0.39090,0.57633,0.34010,0.48739,0.685455,0.827762,a.26.1.1,a.25.2.0,0.14729,P41159,P9WP99
114,1ax8A,2nt8A,0.44627,0.56971,0.33863,0.43118,0.671843,0.836236,a.26.1.1,a.25.2.0,0.09255,P41159,Q50EJ2
115,1ax8A,1yuxA,0.54628,0.65000,0.38044,0.45578,0.678258,0.920509,a.26.1.1,g.41.5.1,0.07534,P41159,P30820


In [55]:
df[df['Protein1_sccs'] == df['Protein2_sccs']]

,Protein1,Protein2,tmscore_max,tmscore_cp_max,tmscore_min,tmscore_cp_min,progres_score,cirpin_score,Protein1_sccs,Protein2_sccs,tm_score_min_difference,Protein1_uniprot,Protein2_uniprot
537,1l3pA,1nlxA,0.62705,0.67800,0.61776,0.66790,0.990689,0.989440,a.24.17.1,a.24.17.1,0.05014,Q40963,P43215
820,1jf0A,1prwA,0.45598,0.52366,0.45598,0.52366,0.973755,0.979054,a.39.1.5,a.39.1.5,0.06768,Q27709,P62157
825,1nyaA,1prwA,0.41997,0.58432,0.41997,0.58432,0.967175,0.979064,a.39.1.5,a.39.1.5,0.16435,P06495,P62157
828,1prwA,2sasA,0.55797,0.68087,0.47243,0.57017,0.963018,0.974341,a.39.1.5,a.39.1.5,0.09774,P62157,P04570
830,1qv0A,1dtlA,0.53604,0.58661,0.53604,0.58661,0.959814,0.970897,a.39.1.5,a.39.1.5,0.05057,Q27709,P09860
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4154,1ti1A,1un2A,0.48785,0.93896,0.48785,0.93896,0.991267,0.995208,c.47.1.13,c.47.1.13,0.45111,P0AEG4,P0AEG4
4155,1un2A,1bq7A,0.51353,0.96431,0.51353,0.96431,0.989165,0.995577,c.47.1.13,c.47.1.13,0.45078,P0AEG4,P0AEG4
4156,1un2A,2b3sA,0.47834,0.91446,0.47593,0.90986,0.985751,0.993427,c.47.1.13,c.47.1.13,0.43393,P0AEG4,P0AEG4
4159,1yzxA,1un2A,0.30365,0.56550,0.30365,0.56550,0.976762,0.971317,c.47.1.13,c.47.1.13,0.26185,Q9Y2Q3,P0AEG4
